In [49]:
import os
import zipfile
import tempfile
import shutil
from collections import defaultdict
from datetime import datetime
from typing import Optional, Literal

import pandas as pd

In [50]:
OUTER_ZIP_PATH = r"F:\현일\데이터 및 문제\데이터분석 분야_로우데이터.zip"
KIKMIX_PATH    = r"D:\PP\BC\data\og\KIKmix_20230701.csv"
OUT_DIR        = r"F:\현일\extract_gyeongju_agg"
os.makedirs(OUT_DIR, exist_ok=True)

OD_AGG_OUT   = os.path.join(OUT_DIR, "gyeongju_od_agg.parquet")
STAY_AGG_OUT = os.path.join(OUT_DIR, "gyeongju_stay_agg.parquet")

HOLIDAY_DATE = set([
    20230902, 20230903, 20230909, 20230910, 20230916, 20230917, 20230923,
    20230924, 20230928, 20230929, 20230930, 20231001, 20231002, 20231003,
    20231007, 20231008, 20231009, 20231014, 20231015
])

In [51]:
# date -> weekday / is_holiday, time -> hour, gender/age/purpose 그룹핑
WEEKDAY_NAME = {0:"Mon", 1:"Tue", 2:"Wed", 3:"Thu", 4:"Fri", 5:"Sat", 6:"Sun"}

def derive_weekday_and_holiday(date_series: pd.Series):
    """
    date_series: '20230901' or 20230901
    return: weekday('Mon'~'Sun'), is_holiday(0/1)
    """
    d_int = pd.to_numeric(date_series, errors="coerce").astype("Int64")
    dt = pd.to_datetime(d_int.astype("string"), format="%Y%m%d", errors="coerce")
    weekday = dt.dt.weekday.map(WEEKDAY_NAME).fillna("NA").astype("string")
    is_holiday = d_int.isin(HOLIDAY_DATE).astype("int8")
    return weekday, is_holiday

def derive_hour_from_time(t: pd.Series) -> pd.Series:
    """
    '18:00' -> 18 (int16), 결측은 -1
    """
    hh = pd.to_numeric(t.astype("string").str.slice(0, 2), errors="coerce").astype("Int64")
    return hh.fillna(-1).astype("int16")

# gender: 원천 0/1이 정확히 남/녀 어느 쪽인지 너 데이터에서 이미 확인한 값으로 맞추면 됨.
# (부산 예시에서 1=남성,0=여성처럼 보이지만, 혹시 반대면 여기만 바꾸면 됨)
GENDER_CODE_TO_GRP = {0: "F", 1: "M"}  # 필요시 스왑
def map_gender_grp(g: pd.Series) -> pd.Series:
    gg = pd.to_numeric(g, errors="coerce").astype("Int64")
    return gg.map(GENDER_CODE_TO_GRP).fillna("U").astype("string")

# age code (0~8) -> 대표 나이 -> 4그룹(adole/young/middle/senior)
AGE_CODE_TO_AGE = {
    0: 0,
    1: 10,
    2: 20,
    3: 30,
    4: 40,
    5: 50,
    6: 60,
    7: 70,
    8: 80
}
def map_age_grp(age_code: pd.Series) -> pd.Series:
    a = pd.to_numeric(age_code, errors="coerce").astype("Int64").map(AGE_CODE_TO_AGE)
    def _grp(x):
        if pd.isna(x): return "unknown"
        if x < 20:     return "adole"
        if x < 40:     return "young"
        if x < 60:     return "middle"
        return "senior"
    return a.map(_grp).astype("string")

# purpose code (0~5) -> 라벨 -> 3그룹(residential/workedu/consumption)
PURPOSE_CODE_TO_LABEL = {
    0: "귀가",
    1: "업무",
    2: "학업",
    3: "쇼핑여가",
    4: "기타",
    5: "여행",
}
def map_purpose_grp(p: pd.Series) -> pd.Series:
    lab = pd.to_numeric(p, errors="coerce").astype("Int64").map(PURPOSE_CODE_TO_LABEL)
    def _grp(x):
        if pd.isna(x): return "unknown"
        if x in ("귀가", "기타"):       return "residential"
        if x in ("업무", "학업"):       return "workedu"
        if x in ("쇼핑여가", "여행"):   return "consumption"
        return "unknown"
    return lab.map(_grp).astype("string")

def map_purpose_label(p: pd.Series) -> pd.Series:
    lab = pd.to_numeric(p, errors="coerce").astype("Int64").map(PURPOSE_CODE_TO_LABEL)
    return lab.fillna("unknown").astype("string")

In [52]:
# KIKmix 로드 & 경주시 행정동코드 set 만들기
def load_kikmix(kikmix_path: str) -> pd.DataFrame:
    for enc in ("utf-8-sig", "cp949", "euc-kr"):
        try:
            return pd.read_csv(kikmix_path, dtype={"행정동코드": "string"}, encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(kikmix_path, dtype={"행정동코드": "string"})


kikmix = load_kikmix(KIKMIX_PATH)

In [53]:
need_cols = ["행정동코드", "시도명", "시군구명", "읍면동명"]
missing = [c for c in need_cols if c not in kikmix.columns]
if missing:
    raise ValueError(f"KIKmix에 필요한 컬럼이 없습니다: {missing}")

kikmix = kikmix[need_cols].copy()
kikmix["행정동코드"] = kikmix["행정동코드"].astype("string")

gyeongju_k = kikmix[(kikmix["시도명"] == "경상북도") & (kikmix["시군구명"] == "경주시")].copy()
gyeongju_codes = set(gyeongju_k["행정동코드"].dropna().astype("string").unique().tolist())
print(f"[KIKmix] 경상북도 경주시 행정동코드 개수: {len(gyeongju_codes):,}")

[KIKmix] 경상북도 경주시 행정동코드 개수: 24


In [54]:
# outer zip 내부 inner zip 목록 찾기
def list_inner_zips(outer_zip_path: str, prefix: str) -> list[str]:
    with zipfile.ZipFile(outer_zip_path, "r") as z:
        names = [n for n in z.namelist() if n.startswith(prefix) and n.lower().endswith(".zip")]
    return sorted(names)

OD_INNER_ZIPS_ALL   = list_inner_zips(OUTER_ZIP_PATH, "od_")
STAY_INNER_ZIPS_ALL = list_inner_zips(OUTER_ZIP_PATH, "stay_")
print(f"[FOUND] OD inner zips: {len(OD_INNER_ZIPS_ALL)}")
print(f"[FOUND] STAY inner zips: {len(STAY_INNER_ZIPS_ALL)}")


[FOUND] OD inner zips: 4
[FOUND] STAY inner zips: 3


In [55]:
# outer zip -> inner zip temp -> csv members iterator

def iter_inner_csv_members_tempfile(outer_zip_path: str, inner_zip_name: str, csv_prefix: str):
    with zipfile.ZipFile(outer_zip_path, "r") as outer:
        with outer.open(inner_zip_name) as src:
            fd, tmp_zip_path = tempfile.mkstemp(suffix=".zip")
            os.close(fd)
            try:
                with open(tmp_zip_path, "wb") as f:
                    shutil.copyfileobj(src, f, length=1024 * 1024)

                with zipfile.ZipFile(tmp_zip_path, "r") as inner:
                    members = [m for m in inner.namelist() if m.lower().endswith(".csv")]
                    members = [m for m in members if os.path.basename(m).startswith(csv_prefix)]
                    for m in sorted(members):
                        yield tmp_zip_path, m
            finally:
                if os.path.exists(tmp_zip_path):
                    os.remove(tmp_zip_path)

In [56]:
# 저장 함수(Parquet 우선, 불가 시 gz CSV)
def save_df(df: pd.DataFrame, path: str):
    try:
        df.to_parquet(path, index=False)
        print(f"[SAVE] parquet -> {path} ({len(df):,} rows)")
    except Exception as e:
        alt = os.path.splitext(path)[0] + ".csv.gz"
        df.to_csv(alt, index=False, encoding="utf-8-sig", compression="gzip")
        print(f"[SAVE] parquet 실패({type(e).__name__}), csv.gz로 저장 -> {alt} ({len(df):,} rows)")

### 집계 함수

In [57]:
# 6) OD 집계: dest_hdong_cd가 경주시 코드에 포함되는 것만 (date는 저장 X)
#     output columns:
#       dest_hdong_cd, weekday, time_bin, gender_grp, age_grp,
#       origin_purpose_grp, dest_purpose_grp, modal, is_holiday, od_cnts_sum
OD_USECOLS = [
    "dest_hdong_cd", "date", "start_time",
    "gender", "age", "modal",
    "origin_purpose", "dest_purpose",
    "od_cnts"
]
OD_DTYPES = {
    "dest_hdong_cd": "string",
    "date": "string",
    "start_time": "string",
    "gender": "Int8",
    "age": "Int8",
    "modal": "Int8",
    "origin_purpose": "Int8",
    "dest_purpose": "Int8",
    "od_cnts": "Int32",
}

def aggregate_gyeongju_od_dest(
    outer_zip_path: str,
    inner_zip_names: list[str],
    gyeongju_code_set: set[str],
    out_path: str,
    chunksize: int = 300_000,
):
    acc = defaultdict(int)
    gyeongju_code_set = set(map(str, gyeongju_code_set))

    total_rows, kept_rows = 0, 0

    for inner_zip_name in inner_zip_names:
        print(f"[OD] {inner_zip_name}")
        for tmp_zip_path, csv_member in iter_inner_csv_members_tempfile(
            outer_zip_path, inner_zip_name, csv_prefix="od_"
        ):
            with zipfile.ZipFile(tmp_zip_path, "r") as inner:
                with inner.open(csv_member) as f:
                    for chunk in pd.read_csv(
                        f, usecols=OD_USECOLS, dtype=OD_DTYPES, chunksize=chunksize
                    ):
                        total_rows += len(chunk)

                        dest = chunk["dest_hdong_cd"].astype("string")
                        sub = chunk.loc[dest.isin(gyeongju_code_set)].copy()
                        if sub.empty:
                            continue
                        kept_rows += len(sub)

                        weekday, is_holiday = derive_weekday_and_holiday(sub["date"])
                        time_bin = derive_hour_from_time(sub["start_time"])
                        gender_grp = map_gender_grp(sub["gender"])
                        age_grp = map_age_grp(sub["age"])
                        # origin_purpose_grp = map_purpose_grp(sub["origin_purpose"])
                        origin_purpose_grp = map_purpose_label(sub["origin_purpose"])
                        #dest_purpose_grp   = map_purpose_grp(sub["dest_purpose"])
                        dest_purpose_grp   = map_purpose_label(sub["dest_purpose"])
                        modal = pd.to_numeric(sub["modal"], errors="coerce").fillna(-1).astype("int16")
                        w = pd.to_numeric(sub["od_cnts"], errors="coerce").fillna(0).astype("int64")

                        tmp = pd.DataFrame({
                            "dest_hdong_cd": sub["dest_hdong_cd"].astype("string"),
                            "weekday": weekday,
                            "time_bin": time_bin,
                            "gender_grp": gender_grp,
                            "age_grp": age_grp,
                            "origin_purpose_grp": origin_purpose_grp,
                            "dest_purpose_grp": dest_purpose_grp,
                            "modal": modal,
                            "is_holiday": is_holiday.astype("int8"),
                            "od_cnts": w
                        })

                        g = tmp.groupby(
                            ["dest_hdong_cd","weekday","time_bin","gender_grp","age_grp",
                             "origin_purpose_grp","dest_purpose_grp","modal","is_holiday"],
                            as_index=False,
                            dropna=False
                        )["od_cnts"].sum()

                        for r in g.itertuples(index=False):
                            key = (
                                r.dest_hdong_cd, r.weekday, int(r.time_bin),
                                r.gender_grp, r.age_grp,
                                r.origin_purpose_grp, r.dest_purpose_grp,
                                int(r.modal), int(r.is_holiday)
                            )
                            acc[key] += int(r.od_cnts)

        print(f"  - progress: total_rows={total_rows:,} kept_rows={kept_rows:,} agg_keys={len(acc):,}")

    out = pd.DataFrame(
        [(k[0],k[1],k[2],k[3],k[4],k[5],k[6],k[7],k[8],v) for k,v in acc.items()],
        columns=[
            "dest_hdong_cd","weekday","time_bin","gender_grp","age_grp",
            "origin_purpose_grp","dest_purpose_grp","modal","is_holiday","od_cnts_sum"
        ]
    )

    save_df(out, out_path)
    return out

In [58]:
# 7) STAY 집계: hdong_cd가 경주시 코드에 포함되는 것만 (date는 저장 X)
#     output columns:
#       hdong_cd, weekday, time_bin, gender_grp, age_grp, purpose_grp, stay_cnts_sum
STAY_USECOLS = ["hdong_cd","date","time","gender","age","purpose","stay_cnts"]
STAY_DTYPES = {
    "hdong_cd": "string",
    "date": "string",
    "time": "string",
    "gender": "Int8",
    "age": "Int8",
    "purpose": "Int8",
    "stay_cnts": "Int32",
}

def aggregate_gyeongju_stay(
    outer_zip_path: str,
    inner_zip_names: list[str],
    gyeongju_code_set: set[str],
    out_path: str,
    chunksize: int = 300_000,
):
    acc = defaultdict(int)
    gyeongju_code_set = set(map(str, gyeongju_code_set))

    total_rows, kept_rows = 0, 0

    for inner_zip_name in inner_zip_names:
        print(f"[STAY] {inner_zip_name}")
        for tmp_zip_path, csv_member in iter_inner_csv_members_tempfile(
            outer_zip_path, inner_zip_name, csv_prefix="stay_"
        ):
            with zipfile.ZipFile(tmp_zip_path, "r") as inner:
                with inner.open(csv_member) as f:
                    for chunk in pd.read_csv(
                        f, usecols=STAY_USECOLS, dtype=STAY_DTYPES, chunksize=chunksize
                    ):
                        total_rows += len(chunk)

                        h = chunk["hdong_cd"].astype("string")
                        sub = chunk.loc[h.isin(gyeongju_code_set)].copy()
                        if sub.empty:
                            continue
                        kept_rows += len(sub)

                        weekday, _ = derive_weekday_and_holiday(sub["date"])
                        time_bin = derive_hour_from_time(sub["time"])
                        gender_grp = map_gender_grp(sub["gender"])
                        age_grp = map_age_grp(sub["age"])
                        #purpose_grp = map_purpose_grp(sub["purpose"])
                        purpose_grp = map_purpose_label(sub["purpose"])
                        w = pd.to_numeric(sub["stay_cnts"], errors="coerce").fillna(0).astype("int64")

                        tmp = pd.DataFrame({
                            "hdong_cd": sub["hdong_cd"].astype("string"),
                            "weekday": weekday,
                            "time_bin": time_bin,
                            "gender_grp": gender_grp,
                            "age_grp": age_grp,
                            "purpose_grp": purpose_grp,
                            "stay_cnts": w
                        })

                        g = tmp.groupby(
                            ["hdong_cd","weekday","time_bin","gender_grp","age_grp","purpose_grp"],
                            as_index=False,
                            dropna=False
                        )["stay_cnts"].sum()

                        for r in g.itertuples(index=False):
                            key = (r.hdong_cd, r.weekday, int(r.time_bin), r.gender_grp, r.age_grp, r.purpose_grp)
                            acc[key] += int(r.stay_cnts)

        print(f"  - progress: total_rows={total_rows:,} kept_rows={kept_rows:,} agg_keys={len(acc):,}")

    out = pd.DataFrame(
        [(k[0],k[1],k[2],k[3],k[4],k[5],v) for k,v in acc.items()],
        columns=["hdong_cd","weekday","time_bin","gender_grp","age_grp","purpose_grp","stay_cnts_sum"]
    )

    save_df(out, out_path)
    return out

In [59]:
# 8) 실행
od_agg = aggregate_gyeongju_od_dest(
    outer_zip_path=OUTER_ZIP_PATH,
    inner_zip_names=OD_INNER_ZIPS_ALL,
    gyeongju_code_set=gyeongju_codes,
    out_path=OD_AGG_OUT,
    chunksize=300_000
)

stay_agg = aggregate_gyeongju_stay(
    outer_zip_path=OUTER_ZIP_PATH,
    inner_zip_names=STAY_INNER_ZIPS_ALL,
    gyeongju_code_set=gyeongju_codes,
    out_path=STAY_AGG_OUT,
    chunksize=300_000
)

print(od_agg.head())
print(stay_agg.head())

[OD] od_20230901_10.zip
  - progress: total_rows=35,827,801 kept_rows=253,846 agg_keys=90,409
[OD] od_20230911_20.zip
  - progress: total_rows=71,278,387 kept_rows=474,635 agg_keys=128,061
[OD] od_20230921_30.zip
  - progress: total_rows=107,046,512 kept_rows=778,720 agg_keys=177,080
[OD] od_20231001_15.zip
  - progress: total_rows=158,838,955 kept_rows=1,223,332 agg_keys=233,428
[SAVE] parquet 실패(ImportError), csv.gz로 저장 -> F:\현일\extract_gyeongju_agg\gyeongju_od_agg.csv.gz (233,428 rows)
[STAY] stay_20230901_15.zip
  - progress: total_rows=56,405,667 kept_rows=386,458 agg_keys=93,736
[STAY] stay_20230916_30.zip
  - progress: total_rows=112,714,911 kept_rows=769,885 agg_keys=96,267
[STAY] stay_20231001_15.zip
  - progress: total_rows=169,352,478 kept_rows=1,159,770 agg_keys=97,732
[SAVE] parquet 실패(ImportError), csv.gz로 저장 -> F:\현일\extract_gyeongju_agg\gyeongju_stay_agg.csv.gz (97,732 rows)
  dest_hdong_cd weekday  time_bin gender_grp age_grp origin_purpose_grp  \
0    4713025000     F

In [60]:
od_agg.shape

(233428, 10)

In [61]:
od_agg.to_csv(r"C:\Users\legen\Desktop\Lab Project\BC\modeling\od_agg.csv", encoding='utf-8-sig', index_label=False)
stay_agg.to_csv(r"C:\Users\legen\Desktop\Lab Project\BC\modeling\stay_agg.csv", encoding='utf-8-sig', index_label=False)

In [62]:
od_agg.head(3)

,dest_hdong_cd,weekday,time_bin,gender_grp,age_grp,origin_purpose_grp,dest_purpose_grp,modal,is_holiday,od_cnts_sum
0,4713025000,Fri,8,F,young,귀가,기타,0,0,237
1,4713025000,Fri,8,F,young,귀가,여행,0,0,179
2,4713025000,Fri,9,F,adole,귀가,여행,0,0,28
